# Real-data examples: old vs new trended-feature denominator

Purpose of this notebook: pull ~1M real tradelines per bureau (first 10 mapped
chunks each) and show, on concrete examples, how the effective month range
(the denominator of every `percent_<rate>_<window>_months` feature) differs
between the shipping code and the proposed fix.

## The changes this notebook demonstrates

**Asset changes (model-engine, per-bureau FE2 trade.json):**

- Added `missing_data_chars` to each bureau's `PaymentPatternsAggregatorV2`
  params -- the bureau-defined code for "no rating observed this month":
  - Equifax `["*"]` -- "Rate/Status was not available for that month"
    (STS TotalView Programming Guide, p. 3-30)
  - Experian `["-"]` -- "No update received" (CIS Cross Reference Guide,
    Appendix T "Payment Profile Indicators", Segment 357.B4.5, p. 139)
  - TransUnion `["X"]` -- no data received from the subscriber / account in
    dispute (TU4.1 User Guide, Appendix C, pp. 839-840)
- Removed `placeholder: "-"` from the Experian asset: the dash is a real
  month, not formatting, so stripping it shifted every older month one
  position more recent -- misaligning lookback windows and `months_since_*`
  features. The dash now stays in the string and is excluded from denominators
  via `missing_data_chars` instead.
- Removed `placeholder: "/"` from the TransUnion asset: the TU pattern
  character set (`1-5, E, X, J, K, H, G, L, Y`; TU4.1 Appendix C,
  pp. 838-840) has no formatting characters -- the `/` was copied from the
  Equifax asset and never occurs in TU data (verified on these chunks below).
  Only Equifax keeps its placeholder, where `/` genuinely appears as a
  separator after every 12 months of history.

**Code changes (feature-engine-parts, `payment_pattern_aggregator.py`):**

- `exclude_trailing` renamed to `missing_data_chars` and made a constructor
  param fed from the asset (it was hardcoded `["*"]` in `transform`).
- `_get_effective_month_range` gained an `all_month_range` flag: `True`
  subtracts only trailing missing-data runs (legacy semantic, kept for
  `payment_history_length` = account tenure); `False` subtracts every
  occurrence anywhere in the window (spec-correct for trended denominators).
- `_construct_trended_features` anchors the denominator on
  `trimmed.str.len()` instead of the nominal window, so bureaus whose strings
  run shorter than the window aren't inflated by positions that don't exist.
- `_get_count` now wraps values in `re.escape` -- `Series.str.count` compiles
  its pattern as a regex, and Equifax's `*` would raise "nothing to repeat".

**In this notebook** the TU and Experian patterns are combined BY HAND
(combine -> trim -> add fillers, skipping `_remove_placeholder`) because the
installed assets still declare placeholders that are NOT actually placeholders
for those bureaus -- Experian's `-` is a real month and TU's `/` never occurs.
Only Equifax goes through the normal path, since its `/` really is formatting.
This means the Experian patterns here KEEP their dashes, matching the
post-placeholder-removal behavior.

**Note on the input files:** the parquet chunks under
`payment_processing_research_data/<bureau>/test/mapped/` already had the
asset's `mapping` block applied -- MapperV2 ran the StringConverter /
DateConverter / NumericConverter steps from each bureau's FE2 trade asset when
`map_and_save_mapped_data.ipynb` wrote them. So `rptDate`, `date_of_request`,
and the payment-pattern columns are already in their converted form here, and
we do NOT re-run MapperV2: `prep_bureau` below only adds the DateDiff column
and combines the pattern columns into `zest_payment_pattern`.

Denominator definitions being compared (same math as
`analyze_difference_in_denominator.ipynb`):

    eff_old = month_range          - count('#')
    eff_new = len(trimmed_pattern) - count('#') - count(missing_data_char)


In [32]:
import re
from pathlib import Path

import pandas as pd
import numpy as np

from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2
from feature_engine_parts.fe_parts_V2.preprocessors.date_diff import DateDiffV2

from configs import EQUIFAX, EXPERIAN, TRANSUNION, mapped_dir
from helpers import (
    MISSING_DATA_CHARS,
    PROJECT_COLS,
    MONTH_RANGES,
    load_asset_json,
    get_aggregator_params,
)

SPLIT = 'test'

BUREAU_CFGS = {'equifax': EQUIFAX, 'experian': EXPERIAN, 'transunion': TRANSUNION}

In [33]:
def prep_bureau(bureau, trade_df_mapped):
    """Returns to_use_for_payment_processing with `zest_payment_pattern` populated.

    Input must already be MAPPED (the asset's `mapping` block applied) -- the
    mapped/ chunks are, so there is no MapperV2 step here.
    """
    asset = load_asset_json(bureau)
    cols = [c for c in PROJECT_COLS[bureau] if c in trade_df_mapped.columns]
    to_use_for_payment_processing = trade_df_mapped[cols].copy()

    # DateDiffV2: (date_of_request - rptDate) / 30.436875 days -> months_since_rptDate
    date_diff = DateDiffV2(feature='rptDate', reference_feature='date_of_request',
                           new_feature='months_since_rptDate')
    to_use_for_payment_processing = date_diff.transform(to_use_for_payment_processing)

    agg = PaymentPatternsAggregatorV2(**get_aggregator_params(asset))
    if bureau in ('transunion', 'experian'):
        # By hand for TU and Experian, SKIPPING _remove_placeholder: what the
        # installed assets call "placeholder" is NOT a placeholder for these
        # two bureaus -- there is no formatting character to strip.
        #   - experian: '-' is a real month, status 'No update received'
        #     (CIS Cross Reference Guide, Appendix T, Segment 357.B4.5,
        #     p. 139). Stripping it shifts every older month one position
        #     more recent; it stays in the string and is excluded from the
        #     denominators via missing_data_chars instead.
        #   - transunion: the pattern character set (1-5, E, X, J, K, H, G,
        #     L, Y; TU4.1 User Guide, Appendix C, pp. 838-840) has no
        #     formatting chars; the asset's '/' was copied from Equifax and
        #     never occurs in TU data.
        # Same steps as _construct_payment_pattern_cols minus the placeholder.
        new_ppt = agg._combine_payment_patterns(to_use_for_payment_processing)
        new_ppt = agg._trim(new_ppt)
        new_ppt = agg._add_fillers(to_use_for_payment_processing, new_ppt)
        to_use_for_payment_processing['zest_payment_pattern'] = new_ppt
    else:
        # Equifax: combine via the aggregator helper, placeholder removal
        # included -- its '/' really is formatting (a separator after every
        # 12 months of history).
        to_use_for_payment_processing['zest_payment_pattern'] = agg._construct_payment_pattern_cols(
            to_use_for_payment_processing
        )
    return to_use_for_payment_processing

In [34]:
# Load the first N_PARTS mapped chunks per bureau (~100k rows each -> ~1M
# rows per bureau). Paths come from configs.mapped_dir, not hardcoded.
N_PARTS = 10

mapped = {}
for bureau, cfg in BUREAU_CFGS.items():
    d = Path(mapped_dir(cfg, SPLIT))
    parts = sorted(d.glob('part-*.parquet'))[:N_PARTS]
    mapped[bureau] = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    print(f'[{bureau}]  {len(mapped[bureau]):,} rows from {len(parts)} chunks in {d}')

[equifax]  1,000,000 rows from 10 chunks in /home/jag/payment-processor-research/payment_processing_research_data/equifax/test/mapped
[experian]  1,000,000 rows from 10 chunks in /home/jag/payment-processor-research/payment_processing_research_data/experian/test/mapped
[transunion]  1,000,000 rows from 10 chunks in /home/jag/payment-processor-research/payment_processing_research_data/transunion/test/mapped


In [35]:
# Transform all three bureaus: PROJECT_COLS -> DateDiff -> zest_payment_pattern.
prepped = {}
for bureau, df in mapped.items():
    prepped[bureau] = prep_bureau(bureau, df)
    sample = prepped[bureau]['zest_payment_pattern'].dropna()
    print(f'[{bureau}]  {len(prepped[bureau]):,} rows; '
          f'example pattern: {sample.iloc[0][:48] if len(sample) else "(none)"}')

[equifax]  1,000,000 rows; example pattern: #11111111111111*********************************
[experian]  1,000,000 rows; example pattern: ############B00000000000000000000000000000000000
[transunion]  1,000,000 rows; example pattern: #11111111111111111111111111111111111111111111111


## Examples where the denominator changes

For each bureau: compute `eff_old` / `eff_new` for one illustrative window and
show a handful of real tradelines where they disagree, alongside the
`percent_DQ30+` value each method would produce. These are tradelines with
missing-data codes inside the window (or a pattern string shorter than the
window), i.e. exactly the cases the fix targets.

In [36]:
import numpy as np

In [37]:
SHOW_M     = 24   # window to illustrate
N_EXAMPLES = 8    # rows to show per bureau

examples = {}
for bureau, df in prepped.items():
  missing_char = MISSING_DATA_CHARS[bureau]
  dq30_codes   = get_aggregator_params(load_asset_json(bureau))['payment_patterns']['rate']['DQ30+']

  ppt     = df['zest_payment_pattern'].fillna('')
  trimmed = ppt.str[:SHOW_M]

  # OLD: nominal window minus '#' fillers.
  # NEW: observed string length minus '#' and the bureau's missing-data char.
  eff_old = SHOW_M             - trimmed.str.count('#')
  eff_new = trimmed.str.len()  - trimmed.str.count('#') - trimmed.str.count(re.escape(missing_char))

  n_dq30 = trimmed.str.count('|'.join(re.escape(c) for c in dq30_codes))

  # keep EVERY column prep_bureau produced (raw pattern cols, rptDate,
  # date_of_request, months_since_rptDate, zest_payment_pattern, ...)
  # and append the comparison columns on the right.
  out = df.copy()
  out[f'trimmed_{SHOW_M}'] = trimmed
  out['eff_old']           = eff_old
  out['eff_new']           = eff_new
  out['n_DQ30+']           = n_dq30
  out['pct_DQ30+_old']     = (n_dq30 / eff_old).round(4)
  out['pct_DQ30+_new']     = (n_dq30 / eff_new.where(eff_new > 0)).round(4)
  examples[bureau] = out
  out['diff'] = np.abs(out['pct_DQ30+_old']-out['pct_DQ30+_new'])
    
  changed = out[out['eff_old'] != out['eff_new']].sort_values(by= 'diff', ascending = False)
  print(f"\n[{bureau}]  missing_char={missing_char!r}  window={SHOW_M}m  "
        f"{len(changed):,}/{len(out):,} rows change "
        f"({len(changed) / len(out) * 100:.1f}%)")
  # most interesting examples first: a DQ30+ in the window AND a changed denominator
  show = changed.head(N_EXAMPLES)
  with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
      display(show)


[equifax]  missing_char='*'  window=24m  281,257/1,000,000 rows change (28.1%)


,ZEST_KEY,date_of_request,rptDate,RATE_STATUS_CODE,PAYMENT_HISTORY_1_24,PAYMENT_HISTORY_25_36,PAYMENT_HISTORY_37_48,months_since_rptDate,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
618685,00014091333_17,2019-12-31,2019-12-17,Z,************/************,/************,/************,0.459968,#Z***********************************************,#Z**********************,23,1,1,0.0435,1.0,0.9565
657832,00021695393_8,2019-12-31,2019-12-09,6,************/************,/************,/************,0.722807,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
658000,00020918857_3,2019-12-31,2019-12-08,6,************/************,/************,/************,0.755662,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
657967,00021766294_5,2019-12-31,2019-12-01,9,************/************,/************,/************,0.985647,#9***********************************************,#9**********************,23,1,1,0.0435,1.0,0.9565
657928,00002530374_5,2019-12-31,2019-12-01,9,************/************,/************,/************,0.985647,#9***********************************************,#9**********************,23,1,1,0.0435,1.0,0.9565
657916,00016294306_1,2019-12-31,2019-12-27,6,************/************,/************,/************,0.131420,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
657911,00006337179_3,2019-12-31,2019-12-09,6,************/************,/************,/************,0.722807,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
657885,00024681257_8,2019-12-31,2019-12-09,6,************/************,/************,/************,0.722807,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565



[experian]  missing_char='-'  window=24m  278,721/1,000,000 rows change (27.9%)


,ZEST_KEY,date_of_request,rptDate,PAYMENT_PROFILE,months_since_rptDate,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
933341,2178ac7368027f332a6046390fe93ca3-0eabebca2e70a5ceb8d328bc191d5847,2019-12-31,2019-12-02,G,0.952792,#G,#G,23,1,1,0.0435,1.0,0.9565
461270,0f9d78025c8a6e2eddbc883481645f1d-95163f15227ea51c2d70e64e765a2579,2019-12-31,2019-12-02,G,0.952792,#G,#G,23,1,1,0.0435,1.0,0.9565
579422,d488b1db717f66a78ecc1f225c05f64b-ee6df30d46d2513211954e84619f1cb4,2019-12-31,2019-12-13,G,0.591388,#G,#G,23,1,1,0.0435,1.0,0.9565
121205,656de1568b69f6cac12c46b2b294806d-47b86d46b1c18b928e3b433b756956fb,2019-12-31,2019-12-16,G,0.492823,#G,#G,23,1,1,0.0435,1.0,0.9565
412597,a06784be89f36c45f4a64210ea76cfbd-49dba8ed2506d5cd4b23b77d37241e62,2019-12-31,2019-12-26,G,0.164274,#G,#G,23,1,1,0.0435,1.0,0.9565
512173,0e55869d377312068796cd2334a76113-c07a52c326e7a499ef9b3a473eef2ec4,2019-12-31,2019-12-16,G,0.492823,#G,#G,23,1,1,0.0435,1.0,0.9565
549106,6a00784ed74e6734726eb818db4730be-f3bc055bf66a177ee3d459cd50f0bdba,2019-12-31,2019-12-14,G,0.558533,#G,#G,23,1,1,0.0435,1.0,0.9565
50880,51b196730db775ec5359f74c0a2e7734-74f37bb0717db14741959c3a88227e5f,2019-12-31,2019-12-21,G,0.328549,#G,#G,23,1,1,0.0435,1.0,0.9565



[transunion]  missing_char='X'  window=24m  257,902/1,000,000 rows change (25.8%)


,ZEST_KEY,date_of_request,rptDate,ppt_status,PAYMENT_PATTERN,months_since_rptDate,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
505557,10951351_11039466384,2019-12-31,2019-12-01,L,None,0.985647,#L,#L,23,1,1,0.0435,1.0,0.9565
229667,15529877_11007547622,2019-12-31,2019-12-05,L,None,0.854227,#L,#L,23,1,1,0.0435,1.0,0.9565
229963,13972843_11037263993,2019-12-31,2019-12-08,G,None,0.755662,#G,#G,23,1,1,0.0435,1.0,0.9565
832932,6535219_10975733002,2019-12-31,2019-12-19,L,None,0.394259,#L,#L,23,1,1,0.0435,1.0,0.9565
229813,16998708_10958573627,2019-12-31,2019-12-04,L,None,0.887082,#L,#L,23,1,1,0.0435,1.0,0.9565
978147,10319469_11052445617,2019-12-31,2019-12-08,G,None,0.755662,#G,#G,23,1,1,0.0435,1.0,0.9565
229738,13972843_11041754170,2019-12-31,2019-12-08,G,None,0.755662,#G,#G,23,1,1,0.0435,1.0,0.9565
833133,21009700_11038341479,2019-12-31,2019-12-02,L,None,0.952792,#L,#L,23,1,1,0.0435,1.0,0.9565


In [38]:
exp = examples['experian']   # or prepped['experian'] if you want pre-comparison columns

has_dash = exp[exp['PAYMENT_PROFILE'].fillna('').str.contains('-', regex=False)]
print(f"{len(has_dash):,}/{len(exp):,} experian rows have '-' in PAYMENT_PROFILE "
    f"({len(has_dash) / len(exp) * 100:.1f}%)")

175,999/1,000,000 experian rows have '-' in PAYMENT_PROFILE (17.6%)


In [39]:
has_dash

,ZEST_KEY,date_of_request,rptDate,PAYMENT_PROFILE,months_since_rptDate,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
12,09607ffd407d56fdbc4460eb75e90bfc-380cbcfa2c09f...,2019-12-31,2019-12-16,CCCCC-CCCCCCCCCCCC,0.492823,#CCCCC-CCCCCCCCCCCC,#CCCCC-CCCCCCCCCCCC,23,17,0,0.0000,0.0000,0.0000
13,13bd3a84d0bb6fd5aee333b625ada66d-c4a1b73c68d92...,2019-12-31,2013-09-05,B00000000000000000000000----------------------...,75.829072,##############################################...,########################,0,0,0,NaN,NaN,NaN
20,e6f4adafa5a59c6dfe736d8d65eee983-63dac1eb47b4b...,2019-12-31,2016-10-17,BCCC1CCCCCCCCCCCCCCCCCCCC1CCCCCCCCCCCCCCCCCCCC...,38.440214,#######################################BCCC1CC...,########################,0,0,0,NaN,NaN,NaN
21,768faf30b66811b17b8b80f745e0c31b-814c36673eca8...,2019-12-31,2018-03-09,9CCCCCCCCCCCCCCCCCCCCCCCCCC--0--000CCCCCCCCCCC...,21.749933,######################9CCCCCCCCCCCCCCCCCCCCCCC...,######################9C,2,2,1,0.5000,0.5000,0.0000
29,a3d2f08606535cce7f246384500e87b8-eae6e12d77a79...,2019-12-31,2019-11-18,C---------------------------------------------...,1.412760,##C-------------------------------------------...,##C---------------------,22,1,0,0.0000,0.0000,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
999923,8351498930dd6e50e15cedf3b2f4bcb2-0f9702f5f3ab4...,2019-12-31,2018-10-31,B-CCCCCCCCCCCCCCCCCCCCCCCCCCCC,13.996181,##############B-CCCCCCCCCCCCCCCCCCCCCCCCCCCC,##############B-CCCCCCCC,10,9,0,0.0000,0.0000,0.0000
999928,f42d1756f2b61e049a810e4538b4e3b2-d9bac948f14cf...,2019-12-31,2019-11-29,CCCC1CCCCCCCCCCC-CCCCCCCCCCCCCCCCCCCCCCCCCCCCC...,1.051356,##CCCC1CCCCCCCCCCC-CCCCCCCCCCCCCCCCCCCCCCCCCCC...,##CCCC1CCCCCCCCCCC-CCCCC,22,21,1,0.0455,0.0476,0.0021
999952,343547b57b523df39dcb297b289b9a95-ea89b444356cc...,2019-12-31,2010-06-24,BCCCCCCCCCCCCCCCCCCCCCCC------0CCCCCC,114.236432,##############################################...,########################,0,0,0,NaN,NaN,NaN
999973,8442a825fcfab601b9c6c46953c198a9-7259e44dd4044...,2019-12-31,2016-04-30,B--CCCCCCCCCC,44.025545,#############################################B...,########################,0,0,0,NaN,NaN,NaN


In [40]:
tu = examples['transunion']   # or prepped['experian'] if you want pre-comparison columns



In [41]:
has_slash = tu[tu['PAYMENT_PATTERN'].fillna('').str.contains('/', regex=False)]

In [42]:
has_slash

,ZEST_KEY,date_of_request,rptDate,ppt_status,PAYMENT_PATTERN,months_since_rptDate,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff


In [43]:
has_y = tu[tu['PAYMENT_PATTERN'].fillna('').str.contains('Y', regex=False)]

In [44]:
has_y

,ZEST_KEY,date_of_request,rptDate,ppt_status,PAYMENT_PATTERN,months_since_rptDate,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
